In [1]:
import os
import random
import json
import time
import zipfile
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler
from tqdm import tqdm
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# ==========================================
# 1. Setup, Configuration & Authentication
# ==========================================
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
# TRAIN_FILE = "Phase_1_Pretraining/Train/train_supreme_court.json"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

PROPORTIONS = {
    "train_supreme_court.json": 0.40,
    "train_high_courts.json": 0.25,
    "train_acts.json": 0.20,
    "train_tribunals.json": 0.15
}

# Hyperparameters
TRAIN_TOKENS = 150_000_000  # Adjust based on your time constraints
VAL_TOKENS = 4_000_000
TRAIN_BLOCK_SIZE = 2048
EPOCHS = 1
SEED = 42
random.seed(SEED)

# Custom Model Dimensions (~50M params)
MODEL_CONTEXT_LENGTH = 2048
N_EMBD = 560
N_LAYER = 15
N_HEAD = 8
N_KV_HEAD = 4 
VOCAB_SIZE = 32000

# Training Optimizations (2x T4 specific)
BATCH_SIZE = 2       # Scaled up for dual GPUs
ACCUMULATION_STEPS = 16 # Effective batch size = 32

# Generation Config
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.7
TOP_P = 0.9

# Paths
OUTPUT_DIR = "/kaggle/working/cs781_phase1"
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "legal-custom-final")
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "latest_checkpoint.pt")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
os.makedirs(OUTPUT_DIR, exist_ok=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


CUDA available: True
GPU count: 2


In [2]:
data_files = {}
for filename in PROPORTIONS.keys():
    print(f"Downloading {filename}...")
    # The training files are located in the Phase_1_Pretraining/Train/ directory
    file_path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=f"Phase_1_Pretraining/Train/{filename}",
        repo_type="dataset",
        token=HF_TOKEN
    )
    data_files[filename] = file_path

Phase_1_Pretraining/Train/train_supreme_(…): reconstructing file:   0%|          |  0.00B / 1.32GB            

Phase_1_Pretraining/Train/train_supreme_(…): downloading bytes:           |  0.00B            

Phase_1_Pretraining/Train/train_high_cou(…): reconstructing file:   0%|          |  0.00B / 10.7GB            

Phase_1_Pretraining/Train/train_high_cou(…): downloading bytes:           |  0.00B            

Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

Phase_1_Pretraining/Train/train_tribunal(…): reconstructing file:   0%|          |  0.00B / 6.48GB            

Phase_1_Pretraining/Train/train_tribunal(…): downloading bytes:           |  0.00B            

In [3]:
# import json
# import os
# import random
# from collections import Counter, defaultdict

# class CustomScratchBPE:
#     def __init__(self, vocab_size, special_tokens=None):
#         self.vocab_size = vocab_size
#         self.special_tokens = special_tokens or ["[UNK]", "[PAD]", "[BOS]", "[EOS]"]
#         self.vocab = {}       
#         self.inv_vocab = {}   
#         self.merges = {}      
        
#     def train(self, text_list):
#         print("Aggregating and filtering word frequencies...")
#         word_freqs = Counter()
        
#         for text in text_list:
#             words = ["Ġ" + w for w in text.split() if w and len(w) <= 35]
#             word_freqs.update(words)
            
#         # Drop singletons to save RAM
#         word_freqs = Counter({word: freq for word, freq in word_freqs.items() if freq > 1})
#         print(f"Unique words after filtering singletons: {len(word_freqs):,}")
            
#         unique_chars = set()
#         for word in word_freqs:
#             unique_chars.update(list(word))
            
#         idx = 0
#         for st in self.special_tokens:
#             self.vocab[st] = idx
#             self.inv_vocab[idx] = st
#             idx += 1
            
#         for ch in sorted(list(unique_chars)):
#             self.vocab[ch] = idx
#             self.inv_vocab[idx] = ch
#             idx += 1
            
#         splits = {word: [self.vocab[c] for c in word] for word in word_freqs}
#         target_merges = self.vocab_size - len(self.vocab)
#         print(f"Base character vocab size: {len(self.vocab)}. Target merges: {target_merges}")
        
#         # ---------------------------------------------------------
#         # SPEED OPTIMIZATION: Initial Pair Counts & Inverted Index
#         # ---------------------------------------------------------
#         pair_counts = defaultdict(int)
#         pair_to_words = defaultdict(set)
        
#         for word, freq in word_freqs.items():
#             split = splits[word]
#             for j in range(len(split) - 1):
#                 pair = (split[j], split[j+1])
#                 pair_counts[pair] += freq
#                 pair_to_words[pair].add(word)

#         for i in range(target_merges):
#             if not pair_counts:
#                 break
                
#             best_pair = max(pair_counts, key=pair_counts.get)
#             new_id = idx
#             idx += 1
            
#             self.merges[best_pair] = new_id
#             new_token_str = self.inv_vocab[best_pair[0]] + self.inv_vocab[best_pair[1]]
#             self.vocab[new_token_str] = new_id
#             self.inv_vocab[new_id] = new_token_str
            
#             # ---------------------------------------------------------
#             # FAST INCREMENTAL UPDATE: Only modify words containing best_pair
#             # ---------------------------------------------------------
#             words_to_update = list(pair_to_words.get(best_pair, []))
            
#             del pair_counts[best_pair]
#             if best_pair in pair_to_words:
#                 del pair_to_words[best_pair]
            
#             for word in words_to_update:
#                 split = splits[word]
#                 freq = word_freqs[word]
                
#                 # 1. Remove old pairs from global tracking for this word
#                 for j in range(len(split) - 1):
#                     pair = (split[j], split[j+1])
#                     if pair != best_pair:
#                         pair_counts[pair] -= freq
#                         if pair_counts[pair] <= 0:
#                             del pair_counts[pair]
#                         if word in pair_to_words[pair]:
#                             pair_to_words[pair].remove(word)

#                 # 2. Perform the merge on this word's split
#                 new_split = []
#                 j = 0
#                 while j < len(split):
#                     if j < len(split) - 1 and (split[j], split[j+1]) == best_pair:
#                         new_split.append(new_id)
#                         j += 2
#                     else:
#                         new_split.append(split[j])
#                         j += 1
#                 splits[word] = new_split

#                 # 3. Add newly created pairs to global tracking for this word
#                 for j in range(len(new_split) - 1):
#                     pair = (new_split[j], new_split[j+1])
#                     pair_counts[pair] += freq
#                     pair_to_words[pair].add(word)
                
#             if (i + 1) % 500 == 0:
#                 print(f"Completed {i + 1}/{target_merges} merges...")

#     def encode(self, text):
#         words = ["Ġ" + w for w in text.split() if w and len(w) <= 35]
#         tokens = []
#         unk_id = self.vocab.get("[UNK]", 0)
        
#         for word in words:
#             split = [self.vocab.get(c, unk_id) for c in word]
#             while len(split) > 1:
#                 min_idx = -1
#                 min_rank = float('inf')
#                 for i in range(len(split) - 1):
#                     pair = (split[i], split[i+1])
#                     if pair in self.merges:
#                         rank = self.merges[pair]
#                         if rank < min_rank:
#                             min_rank = rank
#                             min_idx = i
#                 if min_idx == -1:
#                     break
#                 pair = (split[min_idx], split[min_idx+1])
#                 new_id = self.merges[pair]
#                 split = split[:min_idx] + [new_id] + split[min_idx+2:]
#             tokens.extend(split)
#         return tokens

#     def decode(self, token_ids):
#         text = "".join(self.inv_vocab.get(t, "[UNK]") for t in token_ids)
#         return text.replace("Ġ", " ").strip()

#     def get_vocab_size(self):
#         return len(self.vocab)

#     def save(self, path):
#         serializable_merges = {f"{k[0]},{k[1]}": v for k, v in self.merges.items()}
#         data = {"vocab": self.vocab, "merges": serializable_merges, "special_tokens": self.special_tokens}
#         with open(path, "w", encoding="utf-8") as f:
#             json.dump(data, f, ensure_ascii=False, indent=2)

#     def load(self, path):
#         with open(path, "r", encoding="utf-8") as f:
#             data = json.load(f)
#         self.vocab = data["vocab"]
#         self.inv_vocab = {v: k for k, v in self.vocab.items()}
#         self.special_tokens = data["special_tokens"]
#         self.merges = {tuple(map(int, k.split(","))): v for k, v in data["merges"].items()}

# # ==========================================
# # 2. Load or Train BPE Tokenizer
# # ==========================================
# tokenizer_path = os.path.join(OUTPUT_DIR, "custom_bpe_scratch.json")

# tokenizer = CustomScratchBPE(vocab_size=32000)

# if os.path.exists(tokenizer_path):
#     print(f"Loading existing tokenizer from {tokenizer_path}...")
#     tokenizer.load(tokenizer_path)
#     print(f"Loaded Vocabulary size: {tokenizer.get_vocab_size()}")
# else:
#     print("Streaming subset for tokenizer training...")
#     text_corpus_for_tokenizer = []

#     for filename, filepath in data_files.items():
#         print(f"Adding from {filename}")
#         records = []
        
#         with open(filepath, "r", encoding="utf-8") as f:
#             first_char = f.read(1)
#             f.seek(0)
            
#             if first_char != '[':
#                 for i, line in enumerate(f):
#                     if i > 5000: break
#                     try:
#                         obj = json.loads(line)
#                         if isinstance(obj, dict) and "text" in obj:
#                             records.append(obj)
#                     except:
#                         continue
#             else:
#                 for line in f:
#                     if len(records) >= 2000: break
#                     try:
#                         obj = json.loads(line.strip().strip(','))
#                         if isinstance(obj, dict) and "text" in obj:
#                             records.append(obj)
#                     except:
#                         continue

#         if records:
#             sample_docs = random.sample(records, min(1000, len(records)))
#             text_corpus_for_tokenizer.extend([d["text"] for d in sample_docs if "text" in d])

#     print("Training Custom BPE Tokenizer from scratch (Optimized Incremental)...")
#     tokenizer.train(text_corpus_for_tokenizer)
#     tokenizer.save(tokenizer_path)
#     print(f"Final Vocabulary size: {tokenizer.get_vocab_size()}")

In [4]:
import re
import json
import os
from collections import Counter, defaultdict

def clean_and_filter(text, ascii_threshold=0.85):
    """
    Cleans text and quickly rejects non-English/heavily foreign text.
    """
    if not text or len(text) < 20: 
        return None
        
    ascii_count = sum(1 for c in text if ord(c) < 128)
    if (ascii_count / len(text)) < ascii_threshold:
        return None 

    text = re.sub(r'\s+', ' ', text).strip()
    return text

class CustomByteLevelBPE:
    def __init__(self, vocab_size=32000):
        self.vocab_size = vocab_size
        self.merges = {} 
        
        # Base vocabulary of all 256 individual bytes
        self.vocab = {i: bytes([i]) for i in range(256)}
        self.inverse_vocab = {bytes([i]): i for i in range(256)}
        self.next_id = 256
        
        # Reserved Special Tokens
        self.pad_id = 0  

    def train(self, texts, target_vocab_size):
        print("Aggregating byte sequence frequencies...")
        word_freqs = Counter()
        
        # 1. Chunk texts to prevent massive contiguous memory blocks
        for text in texts:
            # Split into words and whitespace blocks to keep sequence lengths small
            chunks = re.findall(r'\S+|\s+', text)
            for chunk in chunks:
                word_freqs[chunk.encode("utf-8")] += 1
                
        # 2. Drop singletons to massively save RAM
        word_freqs = {w: f for w, f in word_freqs.items() if f > 1}
        print(f"Unique byte sequences to process: {len(word_freqs):,}")

        # Initialize splits: word -> list of byte integers
        splits = {word: list(word) for word in word_freqs}
        
        # 3. Build fast inverted index
        pair_counts = defaultdict(int)
        pair_to_words = defaultdict(set)
        
        for word, freq in word_freqs.items():
            split = splits[word]
            for j in range(len(split) - 1):
                pair = (split[j], split[j+1])
                pair_counts[pair] += freq
                pair_to_words[pair].add(word)

        num_merges = target_vocab_size - 256
        print(f"Starting {num_merges} merges...")
        
        for i in range(num_merges):
            if not pair_counts:
                break
                
            best_pair = max(pair_counts, key=pair_counts.get)
            new_id = self.next_id
            
            # Record the merge
            self.merges[best_pair] = new_id
            self.vocab[new_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
            self.inverse_vocab[self.vocab[new_id]] = new_id
            
            # 4. FAST INCREMENTAL UPDATE: Only process words containing the best_pair
            words_to_update = list(pair_to_words.get(best_pair, []))
            
            del pair_counts[best_pair]
            if best_pair in pair_to_words:
                del pair_to_words[best_pair]
            
            for word in words_to_update:
                split = splits[word]
                freq = word_freqs[word]
                
                # Remove old pairs associated with this word from global tracking
                for j in range(len(split) - 1):
                    pair = (split[j], split[j+1])
                    if pair != best_pair:
                        pair_counts[pair] -= freq
                        if pair_counts[pair] <= 0:
                            del pair_counts[pair]
                        if word in pair_to_words.get(pair, set()):
                            pair_to_words[pair].remove(word)

                # Execute the merge on this word's internal list
                new_split = []
                j = 0
                while j < len(split):
                    if j < len(split) - 1 and (split[j], split[j+1]) == best_pair:
                        new_split.append(new_id)
                        j += 2
                    else:
                        new_split.append(split[j])
                        j += 1
                splits[word] = new_split

                # Add newly formed pairs back to global tracking
                for j in range(len(new_split) - 1):
                    pair = (new_split[j], new_split[j+1])
                    pair_counts[pair] += freq
                    pair_to_words[pair].add(word)
                    
            self.next_id += 1
            
            if (i + 1) % 1000 == 0:
                print(f"Merge {i + 1}/{num_merges} completed.")
                
        print("BPE Training complete!")

    def _get_stats(self, ids):
        counts = Counter()
        for i in range(len(ids) - 1):
            counts[(ids[i], ids[i+1])] += 1
        return counts

    def _merge_vocab(self, ids, pair, idx):
        newids = []
        i = 0
        while i < len(ids):
            if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
                newids.append(idx)
                i += 2
            else:
                newids.append(ids[i])
                i += 1
        return newids

    def encode(self, text):
        # Chunk text during encoding to prevent \(O(N^2)\) slowdowns on long documents
        chunks = re.findall(r'\S+|\s+', text)
        final_ids = []
        
        for chunk in chunks:
            ids = list(chunk.encode("utf-8"))
            while len(ids) >= 2:
                stats = self._get_stats(ids)
                pair = min(stats.keys(), key=lambda p: self.merges.get(p, float("inf")))
                if pair not in self.merges:
                    break 
                ids = self._merge_vocab(ids, pair, self.merges[pair])
            final_ids.extend(ids)
            
        return final_ids

    def decode(self, ids):
        b_text = b"".join(self.vocab[idx] for idx in ids if idx != self.pad_id)
        return b_text.decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return len(self.vocab)

    def save(self, path):
        serializable_merges = {f"{p[0]},{p[1]}": v for p, v in self.merges.items()}
        serializable_vocab = {k: list(v) for k, v in self.vocab.items()}
        data = {
            "vocab_size": self.vocab_size,
            "next_id": self.next_id,
            "merges": serializable_merges,
            "vocab": serializable_vocab
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False)

    def load(self, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            
        self.vocab_size = data["vocab_size"]
        self.next_id = data["next_id"]
        self.merges = {tuple(map(int, k.split(","))): v for k, v in data["merges"].items()}
        self.vocab = {int(k): bytes(v) for k, v in data["vocab"].items()}
        self.inverse_vocab = {v: k for k, v in self.vocab.items()}

# ==========================================
# Execution Block
# ==========================================
tokenizer_path = os.path.join(OUTPUT_DIR, "custom_bpe_scratch.json")
tokenizer = CustomByteLevelBPE(vocab_size=VOCAB_SIZE)

if os.path.exists(tokenizer_path):
    print(f"Loading existing tokenizer from {tokenizer_path}...")
    tokenizer.load(tokenizer_path)
else:
    print("Collecting balanced sample from ALL files for BPE Training...")
    sample_texts = []
    
    for filename, filepath in data_files.items():
        target_lines = int(50000 * PROPORTIONS[filename])
        collected = 0
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                if collected >= target_lines: break
                try:
                    data = json.loads(line.strip().rstrip(','))
                    clean_text = clean_and_filter(data.get("text", ""))
                    if clean_text:
                        sample_texts.append(clean_text)
                        collected += 1
                except: continue
                
    print(f"Total training texts collected: {len(sample_texts)}")
    tokenizer.train(sample_texts, target_vocab_size=VOCAB_SIZE)
    tokenizer.save(tokenizer_path)

Loading existing tokenizer from /kaggle/working/cs781_phase1/custom_bpe_scratch.json...


In [5]:
import gc
gc.collect()

60

In [6]:
import random
import json
import torch

print(f"Allocating memory: Train {TRAIN_TOKENS:,} tokens, Val {VAL_TOKENS:,} tokens...")
train_tokens_tensor = torch.zeros(TRAIN_TOKENS + TRAIN_BLOCK_SIZE, dtype=torch.long, device='cpu')
val_tokens_tensor = torch.zeros(VAL_TOKENS + TRAIN_BLOCK_SIZE, dtype=torch.long, device='cpu')

current_train, current_val = 0, 0

print("-" * 60)
for filename, filepath in data_files.items():
    weight = PROPORTIONS[filename]
    target_train = int(TRAIN_TOKENS * weight)
    target_val = int(VAL_TOKENS * weight)
    file_train, file_val = 0, 0
    
    print(f"Processing: {filename} (Weight: {weight*100:.1f}%)")
    print(f"  Target -> Train: {target_train:,} | Val: {target_val:,}")
    
    with open(filepath, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            # 1. Stop Condition Logging
            if file_train >= target_train and file_val >= target_val: 
                print(f"  [✓] Reached target limits for {filename}. Stopping early.")
                break
                
            # 2. Periodic Progress Logging (Every 50,000 lines)
            if i > 0 and i % 50000 == 0:
                print(f"  ... Read {i:,} lines | Collected Train: {file_train:,} | Val: {file_val:,}")
                
            try:
                data = json.loads(line.strip().rstrip(','))
                clean_text = clean_and_filter(data.get("text", ""))
                if not clean_text: continue
                
                tokens = tokenizer.encode(clean_text)
                if not tokens: continue
                
                tok_tensor = torch.tensor(tokens, dtype=torch.long)
                tok_len = len(tok_tensor)
                
                if file_train >= target_train: is_val = True
                elif file_val >= target_val: is_val = False
                else: is_val = random.random() < 0.05
                
                if is_val:
                    space_left = min(target_val - file_val, len(val_tokens_tensor) - current_val)
                    if space_left > 0:
                        take_len = min(tok_len, space_left)
                        val_tokens_tensor[current_val : current_val + take_len] = tok_tensor[:take_len]
                        current_val += take_len
                        file_val += take_len
                else:
                    space_left = min(target_train - file_train, len(train_tokens_tensor) - current_train)
                    if space_left > 0:
                        take_len = min(tok_len, space_left)
                        train_tokens_tensor[current_train : current_train + take_len] = tok_tensor[:take_len]
                        current_train += take_len
                        file_train += take_len
            except: continue

    # 3. File Completion Logging
    print(f"  [Completed] {filename} -> Final Train: {file_train:,} | Final Val: {file_val:,}")
    print("-" * 60)

print("Formatting blocks...")
train_len = (current_train // TRAIN_BLOCK_SIZE) * TRAIN_BLOCK_SIZE
val_len = (current_val // TRAIN_BLOCK_SIZE) * TRAIN_BLOCK_SIZE

# 4. Final Tensor Shape Logging
print(f"Total valid Train Tokens: {train_len:,} | Total valid Val Tokens: {val_len:,}")

train_blocks = train_tokens_tensor[:train_len].view(-1, TRAIN_BLOCK_SIZE)
val_blocks = val_tokens_tensor[:val_len].view(-1, TRAIN_BLOCK_SIZE)

print(f"Train Blocks Shape: {list(train_blocks.shape)} ({train_blocks.shape[0]:,} sequences)")
print(f"Val Blocks Shape:   {list(val_blocks.shape)} ({val_blocks.shape[0]:,} sequences)")

class BlockDataset(torch.utils.data.Dataset):
    def __init__(self, blocks): self.blocks = blocks
    def __len__(self): return len(self.blocks)
    def __getitem__(self, idx): return self.blocks[idx, :-1], self.blocks[idx, 1:]

train_dataset = BlockDataset(train_blocks)
val_dataset = BlockDataset(val_blocks)
print("Datasets successfully created!")

Allocating memory: Train 150,000,000 tokens, Val 4,000,000 tokens...
------------------------------------------------------------
Processing: train_supreme_court.json (Weight: 40.0%)
  Target -> Train: 60,000,000 | Val: 1,600,000
  [✓] Reached target limits for train_supreme_court.json. Stopping early.
  [Completed] train_supreme_court.json -> Final Train: 60,000,000 | Final Val: 1,600,000
------------------------------------------------------------
Processing: train_high_courts.json (Weight: 25.0%)
  Target -> Train: 37,500,000 | Val: 1,000,000
  [✓] Reached target limits for train_high_courts.json. Stopping early.
  [Completed] train_high_courts.json -> Final Train: 37,500,000 | Final Val: 1,000,000
------------------------------------------------------------
Processing: train_acts.json (Weight: 20.0%)
  Target -> Train: 30,000,000 | Val: 800,000
  [✓] Reached target limits for train_acts.json. Stopping early.
  [Completed] train_acts.json -> Final Train: 30,000,000 | Final Val: 800,

In [7]:
# # ==========================================
# # 3. Tokenization & Dataset Creation (Streaming & Zero-Memory Overhead)
# # ==========================================
# print("Processing and mixing tokens via streaming...")

# train_tokens_tensor = torch.zeros(TRAIN_TOKENS + TRAIN_BLOCK_SIZE, dtype=torch.long, device='cpu')
# val_tokens_tensor = torch.zeros(VAL_TOKENS + TRAIN_BLOCK_SIZE, dtype=torch.long, device='cpu')

# current_train = 0
# current_val = 0

# for filename, filepath in data_files.items():
#     weight = PROPORTIONS[filename]
#     target_train = int(TRAIN_TOKENS * weight)
#     target_val = int(VAL_TOKENS * weight)
#     file_train = 0
#     file_val = 0
    
#     print(f"Tokenizing and streaming from {filename}...")
#     with open(filepath, "r", encoding="utf-8") as f:
#         first_char = f.read(1)
#         f.seek(0)
        
#         # Stream file line-by-line depending on format to protect RAM
#         for i, line in enumerate(f):
#             if i % 15000 == 0 and i > 0:
#                 print(f"[{filename}] Processed {i:,} lines... (Train: {file_train:,} | Val: {file_val:,})")

#             if file_train >= target_train and file_val >= target_val:
#                 break
                
#             try:
#                 if first_char != '[':
#                     data = json.loads(line)
#                 else:
#                     # Clean commas for line-by-line JSON array parsing
#                     stripped = line.strip().rstrip(',')
#                     if stripped in ('[', ']'):
#                         continue
#                     data = json.loads(stripped)
#             except:
#                 continue
                
#             if not isinstance(data, dict) or "text" not in data:
#                 continue
                
#             # --- APPLIED FILTER HERE ---
#             raw_text = data.get("text", "")
#             clean_text = clean_and_filter(raw_text)
            
#             # If the filter rejected it (returned None due to foreign characters), skip it
#             if not clean_text:
#                 continue
                
#             # Encode only the clean, English-heavy text
#             tokens = tokenizer.encode(clean_text)
#             if not tokens:
#                 continue
#             # ---------------------------
                
#             tok_tensor = torch.tensor(tokens, dtype=torch.long)
#             tok_len = len(tok_tensor)
            
#             # Randomly split into validation (5%) or training (95%)
#             if file_train >= target_train:
#                 is_val = True  # Train is full, force into val
#             elif file_val >= target_val:
#                 is_val = False # Val is full, force into train
#             else:
#                 is_val = random.random() < 0.05 # Normal 5% split

#             # Append based on the smart routing using exact slicing
#             if is_val:
#                 space_left = min(target_val - file_val, len(val_tokens_tensor) - current_val)
#                 if space_left > 0:
#                     take_len = min(tok_len, space_left)
#                     val_tokens_tensor[current_val : current_val + take_len] = tok_tensor[:take_len]
#                     current_val += take_len
#                     file_val += take_len
#             else:
#                 space_left = min(target_train - file_train, len(train_tokens_tensor) - current_train)
#                 if space_left > 0:
#                     take_len = min(tok_len, space_left)
#                     train_tokens_tensor[current_train : current_train + take_len] = tok_tensor[:take_len]
#                     current_train += take_len
#                     file_train += take_len
                    
#     print(f"{filename}: Collected {file_train:,} train tokens and {file_val:,} val tokens.")

# # Trim tensors to exact block boundaries and reshape using .view() (zero data copying)
# train_len = (current_train // TRAIN_BLOCK_SIZE) * TRAIN_BLOCK_SIZE
# val_len = (current_val // TRAIN_BLOCK_SIZE) * TRAIN_BLOCK_SIZE

# train_blocks = train_tokens_tensor[:train_len].view(-1, TRAIN_BLOCK_SIZE)
# val_blocks = val_tokens_tensor[:val_len].view(-1, TRAIN_BLOCK_SIZE)

# print(f"Total Training Blocks: {len(train_blocks):,} | Total Validation Blocks: {len(val_blocks):,}")

# class BlockDataset(torch.utils.data.Dataset):
#     def __init__(self, blocks):
#         self.blocks = blocks
#     def __len__(self):
#         return len(self.blocks)
#     def __getitem__(self, idx):
#         return self.blocks[idx, :-1], self.blocks[idx, 1:]

In [8]:
# # ==========================================
# # 3. Tokenization & Dataset Creation (Streaming & Zero-Memory Overhead)
# # ==========================================
# print("Processing and mixing tokens via streaming...")

# # Pre-allocate 1D PyTorch Tensors for training and validation tokens
# train_tokens_tensor = torch.zeros(TRAIN_TOKENS + TRAIN_BLOCK_SIZE, dtype=torch.long,device='cpu')
# val_tokens_tensor = torch.zeros(VAL_TOKENS + TRAIN_BLOCK_SIZE, dtype=torch.long,device='cpu')

# current_train = 0
# current_val = 0

# for filename, filepath in data_files.items():
#     weight = PROPORTIONS[filename]
#     target_train = int(TRAIN_TOKENS * weight)
#     target_val = int(VAL_TOKENS * weight)
#     file_train = 0
#     file_val = 0
    
#     print(f"Tokenizing and streaming from {filename}...")
#     with open(filepath, "r", encoding="utf-8") as f:
#         first_char = f.read(1)
#         f.seek(0)
        
#         # Stream file line-by-line depending on format to protect RAM
#         for i, line in enumerate(f):
#             if i % 15000 == 0 and i > 0:
#                 print(f"[{filename}] Processed {i:,} lines... (Train: {file_train:,} | Val: {file_val:,})")

#             if file_train >= target_train and file_val >= target_val:
#                 break
                
#             try:
#                 if first_char != '[':
#                     data = json.loads(line)
#                 else:
#                     # Clean commas for line-by-line JSON array parsing
#                     stripped = line.strip().rstrip(',')
#                     if stripped in ('[', ']'):
#                         continue
#                     data = json.loads(stripped)
#             except:
#                 continue
                
#             if not isinstance(data, dict) or "text" not in data:
#                 continue
                
#             tokens = tokenizer.encode(data["text"])
#             if not tokens:
#                 continue
                
#             tok_tensor = torch.tensor(tokens, dtype=torch.long)
#             tok_len = len(tok_tensor)
            
#             # Randomly split into validation (5%) or training (95%)
#             if file_train >= target_train:
#                 is_val = True  # Train is full, force into val
#             elif file_val >= target_val:
#                 is_val = False # Val is full, force into train
#             else:
#                 is_val = random.random() < 0.05 # Normal 5% split

#             # Append based on the smart routing
#             # Append based on the smart routing using exact slicing
#             if is_val:
#                 # Calculate exactly how much space is left for this file and global tensor
#                 space_left = min(target_val - file_val, len(val_tokens_tensor) - current_val)
#                 if space_left > 0:
#                     take_len = min(tok_len, space_left)
#                     val_tokens_tensor[current_val : current_val + take_len] = tok_tensor[:take_len]
#                     current_val += take_len
#                     file_val += take_len
#             else:
#                 # Calculate exactly how much space is left for this file and global tensor
#                 space_left = min(target_train - file_train, len(train_tokens_tensor) - current_train)
#                 if space_left > 0:
#                     take_len = min(tok_len, space_left)
#                     train_tokens_tensor[current_train : current_train + take_len] = tok_tensor[:take_len]
#                     current_train += take_len
#                     file_train += take_len
                    
#     print(f"{filename}: Collected {file_train:,} train tokens and {file_val:,} val tokens.")

# # Trim tensors to exact block boundaries and reshape using .view() (zero data copying)
# train_len = (current_train // TRAIN_BLOCK_SIZE) * TRAIN_BLOCK_SIZE
# val_len = (current_val // TRAIN_BLOCK_SIZE) * TRAIN_BLOCK_SIZE

# train_blocks = train_tokens_tensor[:train_len].view(-1, TRAIN_BLOCK_SIZE)
# val_blocks = val_tokens_tensor[:val_len].view(-1, TRAIN_BLOCK_SIZE)

# print(f"Total Training Blocks: {len(train_blocks):,} | Total Validation Blocks: {len(val_blocks):,}")

# class BlockDataset(torch.utils.data.Dataset):
#     def __init__(self, blocks):
#         self.blocks = blocks
#     def __len__(self):
#         return len(self.blocks)
#     def __getitem__(self, idx):
#         return self.blocks[idx, :-1], self.blocks[idx, 1:]

In [9]:

train_loader = torch.utils.data.DataLoader(
    BlockDataset(train_blocks), 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    pin_memory=True, 
    num_workers=2
)
val_loader = torch.utils.data.DataLoader(
    BlockDataset(val_blocks), 
    batch_size=BATCH_SIZE, 
    pin_memory=True
)

print("DataLoaders successfully created!")
print(BATCH_SIZE)

DataLoaders successfully created!
2


In [10]:
# ==========================================
# 4. Model Architecture (Model 4: Sink + Window + Bottleneck + Masking)
# ==========================================
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight

def precompute_rope(dim, max_len, theta=10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(max_len)
    freqs_cis = torch.polar(torch.ones_like(torch.outer(t, freqs).float()), torch.outer(t, freqs).float())
    return torch.view_as_real(freqs_cis)

def apply_rope(xq, xk, freqs_cis):
    xq_complex = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_complex = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    freqs_cis = freqs_cis.view(1, xq_complex.shape[1], 1, xq_complex.shape[-1])
    
    xq_out = torch.view_as_real(xq_complex * freqs_cis).flatten(3).type_as(xq)
    xk_out = torch.view_as_real(xk_complex * freqs_cis).flatten(3).type_as(xk)
    return xq_out, xk_out

class SinkWindowAttention(nn.Module):
    def __init__(self, head_dim, window_size=256, sink_size=4):
        super().__init__()
        self.scale = 1.0 / math.sqrt(head_dim)
        self.window_size = window_size
        self.sink_size = sink_size

    def create_sink_window_mask(self, L_q, L_k, device):
        causal_mask = torch.tril(torch.ones(L_q, L_k, dtype=torch.bool, device=device))
        row_indices = torch.arange(L_q, device=device).unsqueeze(1)
        col_indices = torch.arange(L_k, device=device).unsqueeze(0)
        window_mask = (row_indices - col_indices) < self.window_size
        mask = causal_mask & window_mask
        
        if self.sink_size > 0:
            sink_mask = torch.zeros(L_q, L_k, dtype=torch.bool, device=device)
            sink_mask[:, :self.sink_size] = True
            sink_mask = sink_mask & causal_mask 
            mask = mask | sink_mask
            
        return mask.view(1, 1, L_q, L_k)

    # NEW: Added attention_mask parameter
    def forward(self, q, k, v, is_causal=True, attention_mask=None):
        L_q, L_k = q.shape[2], k.shape[2]
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        
        if is_causal and L_q > 1:
            mask = self.create_sink_window_mask(L_q, L_k, q.device)
            scores = scores.masked_fill(~mask, float('-inf'))
            
        # NEW: Apply the padding mask passed down from the generation loop
        if attention_mask is not None:
            pad_mask = attention_mask.view(attention_mask.size(0), 1, 1, attention_mask.size(1))
            scores = scores.masked_fill(pad_mask == 0, float('-inf'))
            
        weights = torch.softmax(scores, dim=-1)
        return torch.matmul(weights, v)

class EfficientGQA(nn.Module):
    def __init__(self, dim, n_heads, n_kv_heads, window_size=256, sink_size=4):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = dim // n_heads
        
        self.wq = nn.Linear(dim, n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(dim, dim, bias=False)
        
        self.attention = SinkWindowAttention(self.head_dim, window_size, sink_size)

    # NEW: Added attention_mask parameter
    def forward(self, x, freqs_cis, kv_cache=None, attention_mask=None):
        B, L, _ = x.shape
        q = self.wq(x).view(B, L, self.n_heads, self.head_dim)
        k = self.wk(x).view(B, L, self.n_kv_heads, self.head_dim)
        v = self.wv(x).view(B, L, self.n_kv_heads, self.head_dim)
        
        q, k = apply_rope(q, k, freqs_cis)
        
        if kv_cache is not None:
            k_cache, v_cache = kv_cache
            k = torch.cat([k_cache, k], dim=1)
            v = torch.cat([v_cache, v], dim=1)
        
        new_cache = (k, v) 

        k = k.repeat_interleave(self.n_heads // self.n_kv_heads, dim=2)
        v = v.repeat_interleave(self.n_heads // self.n_kv_heads, dim=2)
        
        # NEW: Pass attention_mask to SinkWindowAttention
        out = self.attention(
            q.transpose(1, 2), 
            k.transpose(1, 2), 
            v.transpose(1, 2), 
            is_causal=(L > 1), 
            attention_mask=attention_mask
        )
        out = out.transpose(1, 2).contiguous().view(B, L, -1)
        
        return self.wo(out), new_cache

class StandardFFN(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * 4, bias=False),
            nn.GELU(),
            nn.Linear(dim * 4, dim, bias=False)
        )
    def forward(self, x):
        return self.net(x)

class BottleneckFFN(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim // 4, bias=False),
            nn.GELU(),
            nn.Linear(dim // 4, dim, bias=False)
        )
    def forward(self, x):
        return self.net(x)

class EfficientTransformerBlock(nn.Module):
    def __init__(self, dim, n_heads, n_kv_heads, window_size, sink_size, is_bottleneck=False):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn = EfficientGQA(dim, n_heads, n_kv_heads, window_size, sink_size)
        self.norm2 = RMSNorm(dim)
        
        if is_bottleneck:
            self.ffn = BottleneckFFN(dim)
        else:
            self.ffn = StandardFFN(dim)

    # NEW: Added attention_mask parameter
    def forward(self, x, freqs_cis, kv_cache=None, attention_mask=None):
        # NEW: Pass attention_mask to EfficientGQA
        attn_out, new_cache = self.attn(self.norm1(x), freqs_cis, kv_cache, attention_mask=attention_mask)
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x, new_cache

class CustomLegalModel(nn.Module):
    def __init__(self, vocab_size, dim, n_layers, n_heads, n_kv_heads, max_len, window_size=256, sink_size=4):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, dim)
        
        self.layers = nn.ModuleList([
            EfficientTransformerBlock(
                dim=dim, 
                n_heads=n_heads, 
                n_kv_heads=n_kv_heads, 
                window_size=window_size, 
                sink_size=sink_size, 
                is_bottleneck=(i % 2 != 0)
            ) for i in range(n_layers)
        ])
        
        self.norm = RMSNorm(dim)
        self.output = nn.Linear(dim, vocab_size, bias=False)
        self.output.weight = self.embeddings.weight 
        
        self.register_buffer("freqs_cis", precompute_rope(dim // n_heads, max_len))

    # NEW: Added attention_mask parameter
    def forward(self, tokens, start_pos=0, kv_caches=None, attention_mask=None):
        B, L = tokens.shape
        freqs_cis = self.freqs_cis[start_pos : start_pos + L].to(tokens.device)
        freqs_cis = torch.view_as_complex(freqs_cis)
        
        x = self.embeddings(tokens)
        new_caches = []
        
        for i, layer in enumerate(self.layers):
            cache = kv_caches[i] if kv_caches is not None else None
            # NEW: Pass attention_mask to EfficientTransformerBlock
            x, new_cache = layer(x, freqs_cis, cache, attention_mask=attention_mask)
            new_caches.append(new_cache)
            
        logits = self.output(self.norm(x))
        return logits, new_caches

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomLegalModel(VOCAB_SIZE, N_EMBD, N_LAYER, N_HEAD, N_KV_HEAD, MODEL_CONTEXT_LENGTH)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

model.to(device)
if torch.cuda.device_count() > 1:
    print(f"Maximizing compute: Using {torch.cuda.device_count()} GPUs with DataParallel.")
    model = nn.DataParallel(model)

Total trainable parameters: 53,217,360
Maximizing compute: Using 2 GPUs with DataParallel.


In [11]:
!ls /kaggle/working/cs781_phase1/

custom_bpe_scratch.json  latest_checkpoint.pt  legal-custom-final


In [13]:
# Define the path for the plot since it was missing
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")
FINAL_MODEL_PATH = os.path.join(MODEL_OUTPUT_DIR, "pytorch_model.bin")

# ==========================================
# 5. Training Loop with Resume, Plotting, and Bypass
# ==========================================
from torch.optim.lr_scheduler import OneCycleLR

# 1. Check if the final model is already fully trained
if os.path.exists(FINAL_MODEL_PATH):
    print(f"Final model found at {FINAL_MODEL_PATH}! Loading directly and skipping training.")
    checkpoint_data = torch.load(FINAL_MODEL_PATH, map_location=device, weights_only=True)
    
    # Safely load weights whether using DataParallel or not
    if isinstance(model, nn.DataParallel): 
        model.module.load_state_dict(checkpoint_data)
    else: 
        model.load_state_dict(checkpoint_data)

# 2. If no final model exists, run the training pipeline
else:
    print("Final model not found. Starting/Resuming training pipeline...")
    
    optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.1)
    scaler = GradScaler('cuda')

    total_steps = ((len(train_loader) + ACCUMULATION_STEPS - 1) // ACCUMULATION_STEPS) * EPOCHS
    scheduler = OneCycleLR(optimizer, max_lr=5e-4, total_steps=total_steps + 10, pct_start=0.1, anneal_strategy='cos')

    start_epoch, global_step, resume_batch_idx = 0, 0, -1

    # Resume from intermediate checkpoint if it crashed mid-training
    if os.path.exists(CHECKPOINT_PATH):
        print(f"Resuming from mid-training checkpoint: {CHECKPOINT_PATH}")
        checkpoint_data = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
        if isinstance(model, nn.DataParallel): model.module.load_state_dict(checkpoint_data['model_state_dict'])
        else: model.load_state_dict(checkpoint_data['model_state_dict'])
        optimizer.load_state_dict(checkpoint_data['optimizer_state_dict'])
        scaler.load_state_dict(checkpoint_data['scaler_state_dict'])
        if 'scheduler_state_dict' in checkpoint_data: scheduler.load_state_dict(checkpoint_data['scheduler_state_dict'])
        start_epoch, global_step, resume_batch_idx = checkpoint_data['epoch'], checkpoint_data['global_step'], checkpoint_data.get('batch_idx', -1)

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        running_loss = 0.0 
        
        for i, (x, y) in enumerate(progress_bar):
            if epoch == start_epoch and i <= resume_batch_idx: continue
            x, y = x.to(device), y.to(device)

            with autocast('cuda', dtype=torch.float16):
                logits, _ = model(x) 
                # Replaced .view() with .reshape() to prevent non-contiguous memory crashes
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1), label_smoothing=0.1) / ACCUMULATION_STEPS
                
            scaler.scale(loss).backward()
            running_loss += loss.item() * ACCUMULATION_STEPS

            if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                try: scheduler.step()
                except ValueError: pass
                
                optimizer.zero_grad()
                global_step += 1
                current_train_loss = running_loss / (ACCUMULATION_STEPS if (i + 1) != len(train_loader) else ((i + 1) % ACCUMULATION_STEPS or ACCUMULATION_STEPS))
                running_loss = 0.0 
                
                if global_step % 100 == 0:
                    progress_bar.set_postfix({"LR": f"{scheduler.get_last_lr()[0]:.2e}", "Train": f"{current_train_loss:.4f}"})
                    
                if global_step % 500 == 0:
                    state_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
                    torch.save({'epoch': epoch, 'batch_idx': i, 'global_step': global_step, 'model_state_dict': state_dict, 'optimizer_state_dict': optimizer.state_dict(), 'scaler_state_dict': scaler.state_dict(), 'scheduler_state_dict': scheduler.state_dict()}, CHECKPOINT_PATH)

    # Save final model after training completes
    os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
    final_state_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    torch.save(final_state_dict, FINAL_MODEL_PATH)
    print("Training completed successfully! Final model saved.")

Final model found at /kaggle/working/cs781_phase1/legal-custom-final/pytorch_model.bin! Loading directly and skipping training.


In [14]:
test_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename="Phase_1_Pretraining/Test/test.json", repo_type="dataset", token=HF_TOKEN)
with open(test_file_path, "r", encoding="utf-8") as f: test_data = json.load(f)

eval_model = model.module if isinstance(model, nn.DataParallel) else model
eval_model.eval()

def generate_batch_custom(prompts, eval_model, tokenizer, device, target_words=128):
    batch_size = len(prompts)
    encoded_prompts = [tokenizer.encode(p)[-(MODEL_CONTEXT_LENGTH - MAX_GENERATION_TOKENS):] for p in prompts]
    max_len = max(len(p) for p in encoded_prompts)
    
    pad_id = 0 
    padded_inputs = []
    attention_masks = []
    
    for p in encoded_prompts:
        pad_len = max_len - len(p)
        padded_inputs.append([pad_id] * pad_len + p)
        # Mask: 0 for padding, 1 for real tokens
        attention_masks.append([0] * pad_len + [1] * len(p))
        
    input_tensor = torch.tensor(padded_inputs, dtype=torch.long).to(device)
    attention_mask = torch.tensor(attention_masks, dtype=torch.long).to(device)
    generated_tokens = [[] for _ in range(batch_size)]
    
    with torch.no_grad():
        with autocast('cuda', dtype=torch.float16):
            # PREFILL: Pass attention_mask to ignore left-padding
            logits, kv_caches = eval_model(
                input_tensor, 
                start_pos=0, 
                attention_mask=attention_mask
            )
            
        next_token_logits = logits[:, -1, :].float() / TEMPERATURE
        
        # Top-P Sampling
        sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)
        cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        sorted_indices_to_remove = cumulative_probs > TOP_P
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = 0
        indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
        next_token_logits[indices_to_remove] = float('-inf')
        
        probs = F.softmax(next_token_logits, dim=-1)
        probs = probs + 1e-10 
        probs = probs / probs.sum(dim=-1, keepdim=True)
        next_tokens = torch.multinomial(probs, num_samples=1)
        
        for i in range(batch_size): generated_tokens[i].append(next_tokens[i].item())
        current_token_tensor = next_tokens
        current_pos = input_tensor.shape[1]
        
        # AUTOREGRESSIVE GENERATION
        for _ in range(MAX_GENERATION_TOKENS - 1):
            if current_pos >= MODEL_CONTEXT_LENGTH: break
                
            # Update Mask: newly generated tokens are real, so append 1
            new_mask_token = torch.ones((batch_size, 1), dtype=torch.long, device=device)
            attention_mask = torch.cat([attention_mask, new_mask_token], dim=1)
            
            with autocast('cuda', dtype=torch.float16):
                logits, kv_caches = eval_model(
                    current_token_tensor, 
                    start_pos=current_pos, 
                    kv_caches=kv_caches,
                    attention_mask=attention_mask
                )
                
            next_token_logits = logits[:, -1, :].float() / TEMPERATURE
            sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_indices_to_remove = cumulative_probs > TOP_P
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0
            indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
            next_token_logits[indices_to_remove] = float('-inf')
            
            probs = F.softmax(next_token_logits, dim=-1)
            probs = probs + 1e-10 
            probs = probs / probs.sum(dim=-1, keepdim=True)
            next_tokens = torch.multinomial(probs, num_samples=1)
            
            for i in range(batch_size): generated_tokens[i].append(next_tokens[i].item())
            current_token_tensor = next_tokens
            current_pos += 1
            
    predictions = []
    for i in range(batch_size):
        text = tokenizer.decode(generated_tokens[i])
        words = text.split()[:target_words]
        predictions.append(" ".join(words))
    return predictions

predictions_dict = {}
GENERATION_BATCH_SIZE = 1 

items = list(test_data.items())
for start in tqdm(range(0, len(items), GENERATION_BATCH_SIZE), desc="Batch Generation"):
    batch = items[start:start + GENERATION_BATCH_SIZE]
    batch_ids = [doc_id for doc_id, _ in batch]
    batch_prompts = [record["text"] for _, record in batch]
    
    batch_predictions = generate_batch_custom(
        prompts=batch_prompts, eval_model=eval_model, tokenizer=tokenizer, 
        device=device, target_words=TARGET_GENERATION_WORDS
    )
    for doc_id, prediction in zip(batch_ids, batch_predictions):
        predictions_dict[doc_id] = {"text": prediction}

with open(PREDICTION_PATH, "w", encoding="utf-8") as f: json.dump(predictions_dict, f, ensure_ascii=False, indent=2)
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf: zipf.write(PREDICTION_PATH, arcname="prediction.json")
print("Submission created:", SUBMISSION_ZIP)

test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

Batch Generation: 100%|██████████| 1000/1000 [2:05:32<00:00,  7.53s/it] 

Submission created: /kaggle/working/cs781_phase1/submission.zip


In [ ]:
# def generate_batch_custom(prompts, eval_model, tokenizer, device, target_words=128):
#     batch_size = len(prompts)
    
#     encoded_prompts = [tokenizer.encode(p)[-(MODEL_CONTEXT_LENGTH - MAX_GENERATION_TOKENS):] for p in prompts]
#     max_len = max(len(p) for p in encoded_prompts)
    
#     # Left-pad to create a uniform batch tensor
#     pad_id = 0 
#     padded_inputs = []
#     for p in encoded_prompts:
#         pad_len = max_len - len(p)
#         padded_inputs.append([pad_id] * pad_len + p)
        
#     input_tensor = torch.tensor(padded_inputs, dtype=torch.long).to(device)
    
#     generated_tokens = [[] for _ in range(batch_size)]
    
#     with torch.no_grad():
#         with autocast('cuda', dtype=torch.float16):
#             # FIX 1: Remove attention_mask to prevent PyTorch NaN propagation
#             logits, kv_caches = eval_model(
#                 input_tensor, 
#                 start_pos=0
#             )
            
#         # FIX 2: Cast to float32 before dividing by TEMPERATURE to avoid Infinity overflow
#         next_token_logits = logits[:, -1, :].float() / TEMPERATURE
#         TOP_P = 0.90
        
#         sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)
#         cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        
#         sorted_indices_to_remove = cumulative_probs > TOP_P
#         sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
#         sorted_indices_to_remove[..., 0] = 0
        
#         indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
#         next_token_logits[indices_to_remove] = float('-inf')
        
#         probs = F.softmax(next_token_logits, dim=-1)
#         next_tokens = torch.multinomial(probs, num_samples=1)
        
#         for i in range(batch_size):
#             generated_tokens[i].append(next_tokens[i].item())
            
#         current_token_tensor = next_tokens
#         current_pos = input_tensor.shape[1]
        
#         # 2. Generation Phase (Token-by-Token)
#         for _ in range(MAX_GENERATION_TOKENS - 1):
#             if current_pos >= MODEL_CONTEXT_LENGTH:
#                 break
                
#             with autocast('cuda', dtype=torch.float16):
#                 logits, kv_caches = eval_model(
#                     current_token_tensor, 
#                     start_pos=current_pos, 
#                     kv_caches=kv_caches
#                 )
                
#             # FIX 2 (Applied here as well): Cast to float32
#             next_token_logits = logits[:, -1, :].float() / TEMPERATURE
            
#             sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)
#             cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            
#             sorted_indices_to_remove = cumulative_probs > TOP_P
#             sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
#             sorted_indices_to_remove[..., 0] = 0
            
#             indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
#             next_token_logits[indices_to_remove] = float('-inf')
            
#             probs = F.softmax(next_token_logits, dim=-1)
#             next_tokens = torch.multinomial(probs, num_samples=1)
            
#             for i in range(batch_size):
#                 generated_tokens[i].append(next_tokens[i].item())
                
#             current_token_tensor = next_tokens
#             current_pos += 1
            
#     # Decode and truncate to exact word count on the CPU
#     predictions = []
#     for i in range(batch_size):
#         text = tokenizer.decode(generated_tokens[i])
#         words = text.split()[:target_words]
#         predictions.append(" ".join(words))
        
#     return predictions

# # --- Execution ---
# predictions_dict = {}
# GENERATION_BATCH_SIZE = 16  # Your Dual GPUs can process chunks of 16 instantly

# items = list(test_data.items())
# total_examples = len(items)
# print(f"Total test examples: {total_examples}")
# print("Generating predictions in batches...")

# for start in tqdm(range(0, total_examples, GENERATION_BATCH_SIZE), desc="Batch Generation"):
#     batch = items[start:start + GENERATION_BATCH_SIZE]
#     batch_ids = [doc_id for doc_id, _ in batch]
#     batch_prompts = [record["text"] for _, record in batch]
    
#     batch_predictions = generate_batch_custom(
#         prompts=batch_prompts,
#         eval_model=eval_model,
#         tokenizer=tokenizer,
#         device=device,
#         target_words=TARGET_GENERATION_WORDS
#     )
    
#     for doc_id, prediction in zip(batch_ids, batch_predictions):
#         predictions_dict[doc_id] = {"text": prediction}

# # Save and Zip
# PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
# SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")

# with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
#     json.dump(predictions_dict, f, ensure_ascii=False, indent=2)

# with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
#     zipf.write(PREDICTION_PATH, arcname="prediction.json")

# print("Submission created:", SUBMISSION_ZIP)